In [1]:
import numpy as np
from PIL import Image
import cv2

In [2]:
def crop_and_resize(image_path, output_path, crop_bounds, output_size=(600, 600)):
    with Image.open(image_path) as img:
        width, height = img.size

        left = crop_bounds[0][0] * width
        upper = crop_bounds[0][1] * height
        right = crop_bounds[1][0] * width
        lower = crop_bounds[1][1] * height

        cropped_img = img.crop((left, upper, right, lower))
        resized_img = cropped_img.resize(output_size, Image.LANCZOS)

        resized_img.save(output_path, format='PNG')

In [3]:
# 打印图片中所有拥有的颜色
def print_image_colors(image_path):
    with Image.open(image_path) as img:
        img = img.convert('RGB')
        colors = img.getcolors(maxcolors=50)
        
        if colors:
            for count, color in colors:
                print(f"Color: {color}, Count: {count}")
        else:
            print("Too many colors in the image or maxcolors is too small.")

In [4]:
'''真实数据的Label - 共14个
(255, 0, 0):        人工构造物      
(0, 128, 255):      水田            
(255, 193, 191):    耕地
(0, 0, 100):        水域
(255, 255, 0):      草地
(128, 255, 0):      落叶阔叶林
(0, 255, 128):      落叶针叶林
(86, 172, 0):       常绿阔叶林
(0, 172, 86):       常绿针叶林
(128, 100, 0):      裸地
(217, 240, 5):      竹林
(161, 41, 119):     太阳能板
(0, 150, 160):      湿地
(255, 255, 255):    农业温室
'''

'''合成数据的Label
(1.0f, 1.0f, 1.0f): _MOUNTAIN
(0.1f, 0.8f, 0.3f): _FOREST
(0.5f, 0.2f, 0.2f): _CITY
(0.1f, 0.1f, 0.8f): _WATER
'''

'''合并策略
(人工构造物) (耕地) (水田) (太阳能板) (农业温室) ——> _CITY
(草地) (落叶/常绿 阔叶/针叶林) (竹林) (裸地) (湿地) ——> _FOREST
(水域) ——> _WATER
'''

'合并策略\n(人工构造物) (耕地) (水田) (太阳能板) (农业温室) ——> _CITY\n(草地) (落叶/常绿 阔叶/针叶林) (竹林) (裸地) (湿地) ——> _FOREST\n(水域) ——> _WATER\n'

In [5]:
label_mapping = {
    (255, 0, 0): '_CITY',        # 人工构造物
    (0, 128, 255): '_CITY',      # 水田
    (255, 193, 191): '_CITY',    # 耕地
    (161, 41, 119): '_CITY',     # 太阳能板
    (255, 255, 255): '_CITY',    # 农业温室
    (255, 255, 0): '_FOREST',    # 草地
    (128, 255, 0): '_FOREST',    # 落叶阔叶林
    (0, 255, 128): '_FOREST',    # 落叶针叶林
    (86, 172, 0): '_FOREST',     # 常绿阔叶林
    (0, 172, 86): '_FOREST',     # 常绿针叶林
    (128, 100, 0): '_FOREST',    # 裸地
    (217, 240, 5): '_FOREST',    # 竹林
    (0, 150, 160): '_FOREST',    # 湿地
    (0, 0, 100): '_WATER',       # 水域
}

color_mapping = {
    '_CITY':    (0.5, 0.2, 0.2),
    '_FOREST':  (0.1, 0.8, 0.3),
    '_WATER':   (0.1, 0.1, 0.8)
}

In [6]:
def convert_color_mode_P2RGB(path):
    img = Image.open(path)
    if img.mode == 'P':
        img = img.convert('RGB')
    img.save(path)

In [7]:
convert_color_mode_P2RGB('croped_N35E136.png')

In [8]:
test_img = Image.open('croped_N35E136.png')
test_img.mode

'RGB'

In [7]:
def real2synthesis(path):
    label_mapping = {
        (255, 0, 0): '_CITY',        # 人工构造物
        (0, 128, 255): '_CITY',      # 水田
        (255, 193, 191): '_CITY',    # 耕地
        (161, 41, 119): '_CITY',     # 太阳能板
        (255, 255, 255): '_CITY',    # 农业温室
        (255, 255, 0): '_FOREST',    # 草地
        (128, 255, 0): '_FOREST',    # 落叶阔叶林
        (0, 255, 128): '_FOREST',    # 落叶针叶林
        (86, 172, 0): '_FOREST',     # 常绿阔叶林
        (0, 172, 86): '_FOREST',     # 常绿针叶林
        (128, 100, 0): '_FOREST',    # 裸地
        (217, 240, 5): '_FOREST',    # 竹林
        (0, 150, 160): '_FOREST',    # 湿地
        (0, 0, 100): '_WATER',       # 水域
    }

    color_mapping = {
        '_CITY':    (0.5, 0.2, 0.2),
        '_FOREST':  (0.1, 0.8, 0.3),
        '_WATER':   (0.1, 0.1, 0.8)
    }

    img = Image.open(path)
    img_data = np.array(img)

    new_img_data = np.zeros((img_data.shape[0], img_data.shape[1], 3), dtype=np.float32)

    for original_color, synthetic_label in label_mapping.items():
        mask = np.all(img_data == np.array(original_color, dtype=np.uint8), axis=-1)
        new_color = color_mapping[synthetic_label]
        new_img_data[mask] = new_color
    
    new_img_data_uint8 = (new_img_data * 255).astype(np.uint8)
    new_img = Image.fromarray(new_img_data_uint8)
    new_img_path = 'test_real2synthesis.png'
    new_img.save(new_img_path)

In [8]:
def denoise_in_synthesis_img_v1(input_path, output_path, color_mapping, kernel_size=5):
    img = cv2.imread(input_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # 确保图像格式为RGB

    kernel = np.ones((kernel_size, kernel_size), np.uint8)

    for landcover_type, color_val in color_mapping.items():
        # 转换颜色到OpenCV格式
        color_val_255 = tuple(int(c * 255) for c in color_val)

        # 创建掩码
        mask = cv2.inRange(img, color_val_255, color_val_255)

        # 形态学开运算去掉小噪声
        opening = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        # 形态学闭运算填补小洞
        closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)

        img[closing == 255] = color_val_255

    # 将处理后的图像保存
    cleaned_img = Image.fromarray(img)
    cleaned_img.save(output_path)

In [9]:
def denoise_in_synthesis_img_v2(path, kernel_size=5):
    # 读取图像
    img = cv2.imread(path)

    # 定义颜色范围
    city_color = np.array([128, 0, 0], dtype=np.uint8)  # 城市颜色示例
    water_color = np.array([0, 0, 128], dtype=np.uint8)  # 水域颜色示例

    # 定义内核
    kernel = np.ones((kernel_size, kernel_size), np.uint8)

    # 闭运算去除水域中的小噪声
    water_mask = cv2.inRange(img, water_color, water_color)
    water_close = cv2.morphologyEx(water_mask, cv2.MORPH_CLOSE, kernel)

    # 开运算去除城市边缘的小噪声
    city_mask = cv2.inRange(img, city_color, city_color)
    city_open = cv2.morphologyEx(city_mask, cv2.MORPH_OPEN, kernel)

    # 更新原图像中的对应颜色
    img[water_close == 255] = water_color
    img[city_open == 255] = city_color

    # 再次对全图进行闭运算，以确保边缘的小噪声被填充
    img_close = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)

    # 保存处理后的图像
    cleaned_img_path = path.replace('.png', '_cleaned.png')
    cv2.imwrite(cleaned_img_path, img_close)

    return cleaned_img_path

In [15]:
def dilate_specific_color(img_path, color_to_dilate, kernel_size):
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    target_color = np.array(color_to_dilate)
    
    # 创建膨胀的核
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    
    # 颜色范围转换为膨胀
    mask = cv2.inRange(img, target_color, target_color)
    dilated_mask = cv2.dilate(mask, kernel, iterations=1)
    
    # 将膨胀的掩码应用到原图中对应颜色
    img[dilated_mask == 255] = target_color
    
    # 保存膨胀后的图像
    new_img_path = img_path.replace('.png', '_dilated.png')
    cv2.imwrite(new_img_path, img)
    
    return new_img_path

In [16]:
def expand_color_area(img_path, color_to_expand, kernel_size):
    # 读取图像
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)

    # 设定目标颜色，这里color_to_expand应该是一个像[255, 0, 0]这样的BGR颜色值列表
    lower_color = np.array(color_to_expand)
    upper_color = np.array(color_to_expand)
    
    # 创建形态学膨胀的核
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    
    # 创建颜色掩码
    mask = cv2.inRange(img, lower_color, upper_color)
    
    # 对掩码进行膨胀操作
    dilated_mask = cv2.dilate(mask, kernel, iterations=1)
    
    # 应用膨胀的掩码，将原图中对应的颜色区域扩大
    img[dilated_mask == 255] = color_to_expand
    
    # 保存修改后的图像
    new_img_path = img_path.replace('.png', '_expanded_1.png')
    cv2.imwrite(new_img_path, img)
    
    return new_img_path

In [ ]:
# Water: 204, 25, 25
denoise_in_synthesis_img_v2('clean_croped_N35E136', 10)

In [17]:
dilate_specific_color('clean_croped_N35E136.png', [204, 25, 25], 6)

'clean_croped_N35E136_dilated.png'

In [14]:
expand_color_area("clean_croped_N35E136.png", [204, 25, 25], 10)

'clean_croped_N35E136_expanded.png'